# Imports

In [ ]:
import time

import cv2

import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torch.nn.functional as F

from DepthAnythingV2.depth_anything_v2.dpt import DepthAnythingV2

# reference to cloned repository
from data.DeepFurniture.deepfurniture import DeepFurnitureDataset


DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# Preprocessing & Dataset

In [ ]:
from dataset import DepthAnythingFurnitureDataset

# Model

In [ ]:
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'DepthAnythingV2/checkpoints/depth_anything_v2_{encoder}.pth', map_location='cpu'))
model = model.to(DEVICE).eval()

# Training

In [ ]:
import time
from tqdm import tqdm

def collate_fn(batch):
    return [item for item in batch if item is not None]

train_dataset = DepthAnythingFurnitureDataset(
    DeepFurnitureDataset("./data/DeepFurniture/uncompressed_data")
)
train_loader = DataLoader(
    train_dataset,
    batch_size=16, # vitl: 2, vits: 16
    shuffle=True,
    collate_fn=collate_fn
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
model.to(DEVICE)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    start_time = time.time()

    print(f"\n🚀 Starting Epoch {epoch+1}/{num_epochs}")
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

    for batch in progress_bar:
        images = torch.stack([b["image"] for b in batch]).to(DEVICE)
        depths = torch.stack([b["depth"].squeeze(0) for b in batch]).to(DEVICE)

        pred = model(images)
        loss = F.l1_loss(pred, depths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(train_loader)
    elapsed = time.time() - start_time
    print(f"✅ Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f} | Time: {elapsed:.2f}s")


# Evaluation

In [ ]:
# raw_img = cv2.imread('C:\\Users\\timon\\git\\HSLU.DSPRO2.Beyond2D\\data\\DeepFurniture\\uncompressed_data\\scenes\\DVD3HFDYEJI4OKYQBT3WKSY8\\image.jpg')
# depth = model.infer_image(raw_img) # HxW raw depth map in numpy

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(12, 8))

# axes[0].set_title('Raw Image', fontsize=14)
# axes[0].imshow(raw_img)

# axes[1].set_title('Depth Anything V2', fontsize=14)
# axes[1].imshow(depth)